In [2]:
print("🔄 [1/3] Imports, config, preprocessing...")
import os, cv2, random, warnings, gc
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")

!pip install -q timm einops

import torch
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ── GPU setup ──────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✔ PyTorch device: {device}")

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()

# ── Config ─────────────────────────────────────────────────
HEIGHT, WIDTH    = 384, 384
EPOCHS_VIT       = 15  # MUST match Cell 2
LR_VIT           = 3e-5
CACHE_DIR        = 'preprocessed_cache_384'

# ── Load CSV ────────────────────────────────────────────────
df = pd.read_csv("/kaggle/input/competitions/aptos2019-blindness-detection/train.csv")
df.columns = ["id_code", "diagnosis"]
df["image_path"] = "/kaggle/input/competitions/aptos2019-blindness-detection/train_images/" + df["id_code"] + ".png"

# ── Preprocessing helpers ───────────────────────────────────
def crop_image(img, tol=7):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    return img[np.ix_(mask.any(1), mask.any(0))] if mask.any() else img

def circle_crop(img):
    img = crop_image(img)
    h, w, _ = img.shape; side = max(h, w)
    img = cv2.resize(img, (side, side))
    x, y = side//2, side//2; r = min(x, y)
    mask = np.zeros((side, side), np.uint8)
    cv2.circle(mask, (x, y), r, 1, -1)
    return crop_image(cv2.bitwise_and(img, img, mask=mask))

def preprocess_image(img_path):
    if not os.path.exists(img_path):
        return np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    image = circle_crop(image)
    image = cv2.resize(image, (HEIGHT, WIDTH))
    image = cv2.addWeighted(image, 4, cv2.GaussianBlur(image, (0,0), 10), -4, 128)
    return image

def cache_preprocessed_images(df, cache_dir=CACHE_DIR):
    os.makedirs(cache_dir, exist_ok=True)
    def process_single(idx_row):
        idx, row = idx_row
        cache_path = os.path.join(cache_dir, f"{row['id_code']}.npy")
        if not os.path.exists(cache_path):
            np.save(cache_path, preprocess_image(row['image_path']))
        return cache_path
    with ThreadPoolExecutor(max_workers=4) as ex:
        df['cache_path'] = list(tqdm(ex.map(process_single, df.iterrows()), total=len(df)))
    return df

print("💾 Caching images at 384x384 (runs once ~3min)...")
df = cache_preprocessed_images(df)

X_train, X_val = train_test_split(df, test_size=0.2, random_state=42, stratify=df["diagnosis"])
X_train = X_train.reset_index(drop=True)
X_val   = X_val.reset_index(drop=True)
print(f"✔ Train: {len(X_train)}, Val: {len(X_val)}")
print(f"✔ Class distribution train: {X_train['diagnosis'].value_counts().sort_index().values}")
print(f"✔ Class distribution val: {X_val['diagnosis'].value_counts().sort_index().values}")

🔄 [1/3] Imports, config, preprocessing...
✔ PyTorch device: cuda
💾 Caching images at 384x384 (runs once ~3min)...


100%|██████████| 3662/3662 [08:10<00:00,  7.46it/s]

✔ Train: 2929, Val: 733
✔ Class distribution train: [1444  296  799  154  236]
✔ Class distribution val: [361  74 200  39  59]


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from timm.models.vision_transformer import resize_pos_embed  
from torchvision.transforms import v2 as T
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
import random
import time
import gc
import os
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score
from scipy.optimize import minimize

print("🏗 Building & training RETFound-Green ViT (FINAL)...")

BATCH_SIZE_TORCH = 8
LR_VIT = 3e-5
EPOCHS_VIT = 15

# ── Dataset ──────────────────────────────────────────────────
class APTOSDataset(Dataset):
    def __init__(self, df, augment=False):
        self.cache_paths = df['cache_path'].values
        self.labels = df['diagnosis'].values.astype(np.float32)
        self.augment = augment
        self.normalize = T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        img = np.load(self.cache_paths[idx]).astype(np.float32) / 255.0
        img = torch.from_numpy(img).permute(2, 0, 1)

        if self.augment:
            img = T.RandomResizedCrop(384, scale=(0.8, 1.0))(img)
            img = T.GaussianBlur(3, sigma=(0.1, 2.0))(img)
            if random.random() > 0.5:
                img = torch.flip(img, [2])          # horizontal flip
            if random.random() > 0.5:
                img = torch.flip(img, [1])          # vertical flip
            k = random.randint(0, 3)
            if k > 0:
                img = torch.rot90(img, k, [1, 2])   # 90/180/270
            img = T.RandomAffine(degrees=15, translate=(0.1, 0.1))(img)
            img = T.ColorJitter(brightness=0.3, contrast=0.2)(img)

        img = self.normalize(img)
        return img, torch.tensor(self.labels[idx])

train_loader = DataLoader(
    APTOSDataset(X_train, True),
    batch_size=BATCH_SIZE_TORCH,
    shuffle=True,
    num_workers=4,
    pin_memory=True   # helps if on GPU
)
val_loader = DataLoader(
    APTOSDataset(X_val, False),
    batch_size=BATCH_SIZE_TORCH,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# ── Model ─────────────────────────────────────────────────────
class GreenRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        ckpt_path = "/kaggle/input/datasets/siriusspec/retfound-green/retfoundgreen_statedict.pth"
        if not os.path.exists(ckpt_path):
            raise FileNotFoundError("❌ Add RETFound-Green dataset")

        self.vit = timm.create_model('vit_small_patch14_reg4_dinov2',
                                      img_size=384, num_classes=0)
        state = torch.load(ckpt_path, map_location='cpu')
        if 'model' in state: state = state['model']
        state.pop('head.weight', None)
        state.pop('head.bias', None)

        # YOUR ROBUST INTERP BLOCK
        if 'pos_embed' in state:
            ckpt_pos = state['pos_embed']
            model_pos = self.vit.pos_embed

            if ckpt_pos.shape!= model_pos.shape:
                print(f"⚠️ Resizing pos_embed {ckpt_pos.shape} → {model_pos.shape}")

                # --- Step 1: detect prefix tokens ---
                num_extra_tokens = model_pos.shape[1] - self.vit.patch_embed.num_patches

                # Split tokens
                extra_tokens_ckpt = ckpt_pos[:, :num_extra_tokens]
                pos_tokens_ckpt = ckpt_pos[:, num_extra_tokens:]

                # --- Step 2: compute grid sizes safely ---
                num_patches_ckpt = pos_tokens_ckpt.shape[1]
                gs_old = int(round(num_patches_ckpt ** 0.5))
                gs_new = int(self.vit.patch_embed.num_patches ** 0.5)

                if gs_old * gs_old!= num_patches_ckpt:
                    raise ValueError(
                        f"❌ Checkpoint patches not square: {num_patches_ckpt}"
                    )

                # --- Step 3: interpolate ---
                pos_tokens_ckpt = pos_tokens_ckpt.reshape(
                    1, gs_old, gs_old, -1
                ).permute(0, 3, 1, 2)

                pos_tokens_ckpt = torch.nn.functional.interpolate(
                    pos_tokens_ckpt,
                    size=(gs_new, gs_new),
                    mode='bicubic',
                    align_corners=False
                )

                pos_tokens_ckpt = pos_tokens_ckpt.permute(
                    0, 2, 3, 1
                ).reshape(1, gs_new * gs_new, -1)

                # --- Step 4: use model’s prefix tokens ---
                extra_tokens_model = model_pos[:, :num_extra_tokens]

                state['pos_embed'] = torch.cat(
                    (extra_tokens_model, pos_tokens_ckpt), dim=1
                )

                print(f"✔ pos_embed resized to {state['pos_embed'].shape}")

        msg = self.vit.load_state_dict(state, strict=False)
        print("Missing keys:", msg.missing_keys)
        if hasattr(self.vit, "set_grad_checkpointing"):
            self.vit.set_grad_checkpointing(True)
        self.vit.global_pool = 'avg'
        print("✔ RETFound-Green loaded with pos_embed intact")

        self.head = nn.Sequential(
            nn.Linear(384, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.head(self.vit(x)).squeeze(-1) * 4.0

model_vit = GreenRegressor().to(device)

# ── Freeze all except last 3 blocks + norm for warmup ─────────
for name, p in model_vit.vit.named_parameters():
    if not any(b in name for b in ['blocks.9', 'blocks.10', 'blocks.11', 'norm']):
        p.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_vit.parameters()),
    lr=LR_VIT, weight_decay=0.01
)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_VIT)
criterion = nn.SmoothL1Loss(beta=0.5)  # better than MSE for 0-4 ordinal range

# ── Threshold optimizer with bounds (stable, no leakage) ──────
def optimize_thresholds(y_true, y_pred_continuous):
    def kappa_loss(thresh):
        t0, t1, t2, t3 = np.sort(thresh)
        if not (t0 < t1 < t2 < t3): return 1.0
        bins = [0, t0, t1, t2, t3, 5]
        y_pred = np.digitize(y_pred_continuous, bins) - 1
        return -cohen_kappa_score(y_true, y_pred, weights='quadratic')
    bounds = [(0.1, 1.4), (0.6, 2.4), (1.6, 3.4), (2.6, 4.0)]
    result = minimize(kappa_loss, [0.5, 1.5, 2.5, 3.5],
                      method='L-BFGS-B', bounds=bounds)
    return np.sort(result.x)

# ── Training loop ─────────────────────────────────────────────
best_kappa = 0
start_time = time.time()

for epoch in range(EPOCHS_VIT):

    # Stage 1 unfreeze: blocks 9-11 + norm at epoch 5
    if epoch == 5:
        for name, p in model_vit.vit.named_parameters():
            if any(b in name for b in ['blocks.9', 'blocks.10', 'blocks.11', 'norm']):
                p.requires_grad = True
        for g in optimizer.param_groups: g['lr'] = 1e-5
        print("🔓 Unfroze blocks 9-11 + norm, LR -> 1e-5")

    # Stage 2 unfreeze: everything at epoch 10
    if epoch == 10:
        for p in model_vit.parameters(): p.requires_grad = True
        for g in optimizer.param_groups: g['lr'] = 5e-6
        print("🔓 Unfroze all layers, LR -> 5e-6")

    model_vit.train()
    running_loss = 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS_VIT}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_vit(imgs), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_vit.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()

    model_vit.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            preds = model_vit(imgs).cpu().detach().numpy()
            all_preds.extend(preds)
            all_true.extend(labels.numpy())

    all_preds = np.clip(np.array(all_preds), 0, 4)
    all_true = np.array(all_true)
    preds_rounded = np.clip(np.round(all_preds), 0, 4).astype(int)
    kappa = cohen_kappa_score(all_true, preds_rounded, weights='quadratic')
    epoch_time = (time.time() - start_time) / 60
    print(f"Epoch {epoch+1}: loss={running_loss/len(train_loader):.4f} | "
          f"kappa={kappa:.4f} | {epoch_time:.1f}min")

    if kappa > best_kappa:
        best_kappa = kappa
        torch.save(model_vit.state_dict(), 'vit_green_best.pth')
        print(f"💾 Saved best (kappa={kappa:.4f})")

    # Only keep best + latest, don't flood disk
    torch.save({
        'epoch': epoch,
        'model_state_dict': model_vit.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_kappa': best_kappa,
    }, 'vit_green_latest.pth')
    if os.path.exists(f'green_ckpt_e{epoch-1}.pth'):
        os.remove(f'green_ckpt_e{epoch-1}.pth')

    scheduler.step()
    gc.collect()

total_time = (time.time() - start_time) / 60
print(f"\n✔ Done — Best Kappa: {best_kappa:.4f} | Total: {total_time:.1f}min\n")

# ── Save APTOS val thresholds for clean Messidor evaluation ───
print("Computing and saving val thresholds...")
model_vit.load_state_dict(torch.load('vit_green_best.pth'))
model_vit.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        preds = model_vit(imgs).cpu().detach().numpy()
        all_preds.extend(preds)
        all_true.extend(labels.numpy())

all_preds = np.clip(np.array(all_preds), 0, 4)
val_thresh = optimize_thresholds(np.array(all_true), all_preds)
np.save('vit_val_thresholds.npy', val_thresh)
print(f"✔ Val thresholds saved: {val_thresh.round(3)}")
print(f"✔ Best APTOS val kappa: {best_kappa:.4f}")

🏗 Building & training RETFound-Green ViT (FINAL)...
⚠️ Resizing pos_embed torch.Size([1, 784, 384]) → torch.Size([1, 729, 384])
✔ pos_embed resized to torch.Size([1, 729, 384])
Missing keys: []
✔ RETFound-Green loaded with pos_embed intact


Epoch 1/15: 100%|██████████| 367/367 [02:42<00:00,  2.26it/s]


Epoch 1: loss=0.2855 | kappa=0.8631 | 2.9min
💾 Saved best (kappa=0.8631)


Epoch 2/15: 100%|██████████| 367/367 [02:47<00:00,  2.19it/s]


Epoch 2: loss=0.1899 | kappa=0.8795 | 5.9min
💾 Saved best (kappa=0.8795)


...

Epoch 13: loss=0.1164 | kappa=0.9025 | 39.8min


Epoch 14/15: 100%|██████████| 367/367 [03:08<00:00,  1.95it/s]


Epoch 14: loss=0.1203 | kappa=0.9054 | 43.1min


Epoch 15/15: 100%|██████████| 367/367 [03:08<00:00,  1.94it/s]


Epoch 15: loss=0.1166 | kappa=0.9046 | 46.4min

✔ Done — Best Kappa: 0.9087 | Total: 46.5min

Computing and saving val thresholds...
✔ Val thresholds saved: [0.5 1.5 2.5 3.5]
✔ Best APTOS val kappa: 0.9087
